In [1]:
import sys

# ── path setup ──────────────────────────────────────────────────────────────
# Each entry below adds a specific directory so its modules can be imported
# directly (bare name), WITHOUT triggering that directory's __init__.py and
# its full dependency chain. This avoids the two-different-"functions"-
# packages name collision entirely — never `import functions` itself.

sys.path.insert(0, "/home/hcr64/Pinyon-Detection")                                  # global_files/
sys.path.insert(0, "/home/hcr64/Pinyon-Detection/modelling/functions")              # feature_config.py, standalone
sys.path.insert(0, "/home/hcr64/Pinyon-Detection/modelling/functions/features")     # diagnose_cluster_bimodality.py — adjust if saved elsewhere
sys.path.insert(0, "/home/hcr64/Pinyon-Detection/clustering/functions/detection")   # split_large_clusters.py — adjust if saved elsewhere

# ── third-party ────────────────────────────────────────────────────────────
from pyntcloud import PyntCloud
from plyfile import PlyData
import numpy as np
import os
import pandas as pd
import random
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# ── project: paths + cached dataframes/clusters ───────────────────────────
from global_files import load_clusters, load_dataframes, get_paths, save_clusters

# ── project: feature list (standalone, no torch/xgboost/lightgbm pulled in) ─
from feature_config import FEATURES

# ── project: bimodality diagnostics ────────────────────────────────────────
from diagnose_cluster_bimodality import (
    diagnose_all_clusters,
    diagnose_cluster_bimodality,
    calibrate_thresholds,
    explain_why_not_flagged,
    get_color_spatial_override_candidates,
    get_flagged_cluster_paths,
    get_flagged_indices_from_sources,
)

# ── project: colour-augmented splitting ────────────────────────────────────
from split_large_clusters import split_large_clusters, find_density_peaks, filter_cluster

# ── paths + data ────────────────────────────────────────────────────────────
PATHS = get_paths("Sunset_sfm_trial")
df_clusters, df_deep_clusters = load_dataframes(PATHS['Dataframes'])
clusters = load_clusters(PATHS['Clusters'])

print(len(clusters), len(df_clusters))  # sanity check — should match; if not, see the sync-mismatch note from earlier

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


ModuleNotFoundError: No module named 'functions'

In [ ]:
# function to draw a window with a pointcloud in it
def visualize_pointcloud(pcd_path, point_size=2, background_color="black"):

    plydata = PlyData.read(pcd_path)
    vertex = plydata['vertex']

    points = np.vstack([vertex['x'], vertex['y'], vertex['z']]).T

    # use color if available, otherwise default to white
    vertex_names = vertex.data.dtype.names
    if 'red' in vertex_names and 'green' in vertex_names and 'blue' in vertex_names:
        r = vertex['red']
        g = vertex['green']
        b = vertex['blue']
        color_str = [f"rgb({int(r_)},{int(g_)},{int(b_)})" for r_, g_, b_ in zip(r, g, b)]
    else:
        color_str = "white"

    fig = go.Figure(data=[go.Scatter3d(
        x=points[:, 0],
        y=points[:, 1],
        z=points[:, 2],
        mode="markers",
        marker=dict(
            size=point_size,
            color=color_str,
        )
    )])

    fig.update_layout(
        scene=dict(bgcolor=background_color),
        paper_bgcolor="black",
    )

    fig.show()

In [ ]:
# paths and other constants
TRIAL_NAME      = 'Sunset_sfm_trial'
PATHS = get_paths( TRIAL_NAME )

# confirm
print( PATHS['Clusters'] )


In [ ]:
# get the cluster ids of mismatched clusters
df_mismatch = pd.read_csv(PATHS['Dataframes'] + 'pinyon_misclassification_report.csv')

id_col = "file" if "file" in df_mismatch.columns else df_mismatch.columns[0]
mismatched_ids = df_mismatch[id_col].astype(int).tolist()

print(f"{len(mismatched_ids)} mismatched clusters")
print(mismatched_ids)
cluster = 0

In [ ]:
# get the path of desired pointcloud
cluster_id = mismatched_ids[cluster]
PLY_PATH = PATHS['Clusters'] +  f"cluster{cluster_id}.ply"   # change per cluster
cluster += 1

# call the function on it
print( PLY_PATH )
visualize_pointcloud( PLY_PATH )

In [ ]:
PATHS = get_paths("Sunset_sfm_trial")
clusters = load_clusters(PATHS['Clusters'])
labeled_clusters = load_clusters(PATHS['Labeled_clusters'])

df_diag = diagnose_all_clusters(
    clusters,
    save_path=PATHS['Images'] + 'bimodality_diagnostics/',
)

flagged = df_diag.loc[df_diag["color_bimodal_spatial_unimodal"], "file"].unique()
print(sorted(flagged))

In [ ]:
from diagnose_cluster_bimodality import calibrate_thresholds
df_clusters, df_deep_clusters = load_dataframes(PATHS['Dataframes']) 

comparison_indices = [1399, 1375, 1483, 545, 1006, 1744] + list(df_clusters.sample(15, random_state=1)["file"])
df_cal = calibrate_thresholds(clusters, comparison_indices)

In [ ]:
from diagnose_cluster_bimodality import explain_why_not_flagged

explain_why_not_flagged(
    clusters,
    [1399, 1375, 1483, 545, 1006, 1744],
    max_radius=4.25,   # from pinyons.sh, so you can see the production-gate context too
)

In [ ]:
print(len(clusters), len(df_clusters))

In [ ]:
from diagnose_cluster_bimodality import get_color_spatial_override_candidates
#from split_large_clusters import split_large_clusters

override_indices = get_color_spatial_override_candidates(
    clusters, [1399, 1375, 1483, 545, 1006, 1744]
)
# expect [1399, 1375, 1483] back, given what you already saw

test_split = split_large_clusters(
    [clusters[i] for i in override_indices],  # just these 3, for a fast isolated test
    min_points=40, max_radius=4.25, min_peak_distance=3.0,
    use_color=True, color_weight=10.0,
    force_check_indices=set(range(len(override_indices))),  # indices within this small sublist
    save_pre_split_path="/scratch/hcr64/test_override_split/",
)

In [ ]:
from diagnose_cluster_bimodality import get_flagged_cluster_paths

df_diag = diagnose_all_clusters(clusters, ...)  # with your calibrated thresholds

flagged_paths = get_flagged_cluster_paths(df_diag, PATHS['Clusters'])
print(flagged_paths)